# Fine-tuning MobileNetV3-small
        **vision › classification** · notebook `finetune_mobilenet_v3_small`

        **Kapan dipakai:** Laptop tanpa GPU kuat atau batas waktu: model sangat kecil (2.5 juta parameter).

        **Jebakan umum:** Akurasi di bawah model besar; BatchNorm butuh batch cukup.

> ⚠️ Notebook ini berat (butuh torch / bobot / GPU). Lihat sel pertama.

**Data demo:** CIFAR-10 subset (4000 latih / 1000 uji, 10 kelas, 32x32). Bobot pretrained dibaca dari `pretrained/` (offline); GPU dipakai otomatis bila ada.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi
Ganti bagian ini saja saat memakai data lomba. Format data: folder-per-kelas atau CSV `path,label`.

In [ ]:
DATA_ROOT  = ROOT / "data_latihan/vision/classification/cifar10_small"            # folder gambar
TRAIN_CSV  = DATA_ROOT / "train.csv"   # kolom: path (relatif ke DATA_ROOT), label
TEST_CSV   = DATA_ROOT / "test.csv"    # kolom: id, path (tanpa label)
TEST_LABELS = DATA_ROOT / "test_labels.csv"  # kunci jawaban test (hanya untuk latihan); None di lomba
SEED, VALID_SIZE = 42, 0.2
MODEL_KEY = "finetune_mobilenet_v3_small"
OUT = out_dir("vision", "classification", MODEL_KEY)
set_seed(SEED)
SIZE, EPOCHS, LR, BATCH = 64, 8, 0.002, 64
AUG = True                # augmentasi flip/geser/rotasi (GPU)
ARCH = "mobilenet_v3_small"; PRETRAINED = True

## 2. Baca gambar

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from superdb.vision import load_images, show_grid
tr = pd.read_csv(TRAIN_CSV); te = pd.read_csv(TEST_CSV)
le = LabelEncoder(); y = le.fit_transform(tr["label"]); CLASSES = list(le.classes_)
X = load_images(tr["path"], DATA_ROOT); Xte = load_images(te["path"], DATA_ROOT)
print(X.shape, Xte.shape, "| kelas:", CLASSES); show_grid(X[::400], [CLASSES[i] for i in y[::400]], n=10); plt.show()
Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=VALID_SIZE, random_state=SEED, stratify=y)

## 3. Model
Ukuran masukan `SIZE`: gambar kecil diperbesar agar sesuai bobot pretrained (lebih besar = lebih akurat = lebih lambat).

In [ ]:
from pathlib import Path
import torch
from superdb.vision import small_cnn, tv_model, train_classifier, predict_proba, device
print("device:", device())
net = small_cnn(len(CLASSES)) if ARCH == "small_cnn" else tv_model(ARCH, len(CLASSES), pretrained=PRETRAINED)
print(f"{sum(p.numel() for p in net.parameters()) / 1e6:.2f} juta parameter")

## 4. Latih (riwayat per epoch)

In [ ]:
hist = train_classifier(net, Xtr, ytr, Xva, yva, size=SIZE, epochs=EPOCHS, lr=LR, batch_size=BATCH, aug=AUG, seed=SEED)
h = pd.DataFrame(hist); display(h.round(4))
fig, ax = plt.subplots(1, 2, figsize=(9, 3)); h.train_loss.plot(ax=ax[0], marker="o", title="loss latih"); h.valid_acc.plot(ax=ax[1], marker="o", title="akurasi validation"); plt.tight_layout(); plt.savefig(OUT / "curve.png", dpi=130); plt.show()

## 5. Evaluasi

In [ ]:
proba = predict_proba(net, Xva, SIZE); pred = proba.argmax(1)
metrics = classification_metrics(yva, pred, proba, classes=np.arange(len(CLASSES))); display(show_table(metrics))
plot_confusion(yva, pred, labels=np.arange(len(CLASSES)), path=OUT / "confusion.png"); plt.show()

## 6. Prediksi test → submission → simpan bobot
Demi kecepatan, model **tidak** dilatih ulang pada semua data; untuk lomba latih ulang dengan `X, y` penuh setelah memilih konfigurasi.

In [ ]:
CLASSES_PRED = predict_proba(net, Xte, SIZE).argmax(1)

In [ ]:
pred_te = CLASSES_PRED
sub = pd.DataFrame({"id": te["id"], "label": [CLASSES[i] for i in pred_te]}); sub.to_csv(OUT / "submission.csv", index=False); display(sub.head(3))
if TEST_LABELS is not None and Path(TEST_LABELS).exists():
    truth = pd.read_csv(TEST_LABELS); acc = float((truth.set_index("id").loc[sub["id"], "label"].to_numpy() == sub["label"].to_numpy()).mean())
    print(f"akurasi pada test (kunci jawaban): {acc:.4f}"); metrics["test_accuracy"] = acc

In [ ]:
torch.save({"state_dict": net.state_dict(), "classes": CLASSES, "arch": ARCH, "size": SIZE}, OUT / "model.pt")
ck = torch.load(OUT / "model.pt", map_location="cpu"); net2 = (small_cnn(len(CLASSES)) if ARCH == "small_cnn" else tv_model(ARCH, len(CLASSES), pretrained=False)); net2.load_state_dict(ck["state_dict"]); net2.to(device())
assert (predict_proba(net2, Xva[:64], SIZE).argmax(1) == pred[:64]).all(), "model muat-ulang beda"
save_json({"valid": metrics, "history": hist, "config": dict(arch=ARCH, size=SIZE, epochs=EPOCHS, lr=LR, aug=AUG)}, OUT / "metrics.json"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Naikkan `SIZE` (96/128) dan `EPOCHS` untuk skor lebih tinggi; GPU 8 GB cukup untuk batch 64 pada ukuran ini.
- Skor validation jauh di atas test? Cek kebocoran antar split (gambar yang sama/mirip di train dan valid).
- Blend probabilitas beberapa model (mis. fine-tune + CLIP probe) biasanya menambah akurasi.